In [14]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from mtpy import MT, MTData
from loguru import logger
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
edi_path = Path(r"c:\Users\jpeacock\OneDrive - DOI\EDI_FILES\latest_files")

In [16]:
md = MTData()

In [20]:
%%time
edi_list = list(edi_path.glob("*.edi"))
df = pd.DataFrame(index=range(len(edi_list)), 
    columns=["survey", "station", "latitude", "longitude", "file_created", "filename", "period_min", "period_max", "has_impedance", "has_tipper", "note"])
for index, edi_file in enumerate(edi_list):
# for row in failing_df.itertuples():
    # index = row.Index
    # edi_file = edi_path / row.filename
    try:
        logger.info(f"Reading {edi_file.name}")
        mt_obj = MT()
        mt_obj.read(edi_file)
        if mt_obj.station in [0, "0", None]:
            mt_obj.station = edi_file.stem
        df.loc[index, "survey"] = mt_obj.survey
        df.loc[index, "station"] = mt_obj.station
        df.loc[index, "latitude"] = mt_obj.latitude
        df.loc[index, "longitude"] = mt_obj.longitude
        df.loc[index, "file_created"] = edi_file.stat().st_ctime
        df.loc[index, "filename"] = edi_file.name
        df.loc[index, "period_min"] = mt_obj.period.min()
        df.loc[index, "period_max"] = mt_obj.period.max()
        df.loc[index, "has_impedance"] = mt_obj.has_impedance()
        df.loc[index, "has_tipper"] = mt_obj.has_tipper()
        # md.add_station(mt_obj)
    except Exception as e:
        logger.critical(f"Could not read {edi_file.name}: {e}")
        df.loc[index, "note"] = f"Could not read: {e}"
        df.loc[index, "filename"] = edi_file.name

df.to_csv(edi_path.joinpath("edi_file_summary_2026_03_09.csv"), index=False)


26:03:09T10:49:28 | INFO | line:9 |__main__ | <module> | Reading 020_24000.edi
26:03:09T10:49:28 | WARNING | line:432 |mt_metadata.common.units | get_unit_from_df | Unit 'None' not found in accepted units, setting to 'unknown'. If this is an error raise an issue to add a unit. If an error needs to be raised, set allow_none=False.
26:03:09T10:49:28 | INFO | line:9 |__main__ | <module> | Reading 022_24000.edi
26:03:09T10:49:28 | WARNING | line:432 |mt_metadata.common.units | get_unit_from_df | Unit 'None' not found in accepted units, setting to 'unknown'. If this is an error raise an issue to add a unit. If an error needs to be raised, set allow_none=False.
26:03:09T10:49:28 | INFO | line:9 |__main__ | <module> | Reading 023_24000.edi
26:03:09T10:49:28 | WARNING | line:432 |mt_metadata.common.units | get_unit_from_df | Unit 'None' not found in accepted units, setting to 'unknown'. If this is an error raise an issue to add a unit. If an error needs to be raised, set allow_none=False.
26:0

In [21]:
df

,survey,station,latitude,longitude,file_created,filename,period_min,period_max,has_impedance,has_tipper,note
0,VAL,020,35.854478,-106.501775,1772829368.976778,020_24000.edi,0.000186,0.063113,True,True,NaN
1,VAL,022,35.869036,-106.523561,1772829369.136894,022_24000.edi,0.000186,0.063113,True,True,NaN
2,VAL,023,35.937678,-106.574517,1772829369.2392,023_24000.edi,0.000186,0.063113,True,True,NaN
3,VAL,029,35.835714,-106.564928,1772829369.573734,029_24000.edi,0.000186,0.063113,True,True,NaN
4,0,0310,46.933611,-122.199167,1772829369.719573,0310.edi,0.000781,682.687056,True,True,NaN
...,...,...,...,...,...,...,...,...,...,...,...
8574,0,ZTLVC027,1.016944,1.016944,1772829612.665639,ZTLVC027.edi,0.00001,0.063291,True,False,NaN
8575,0,ZTLVC028,1.016944,1.016944,1772829612.685052,ZTLVC028.edi,0.00001,0.063291,True,False,NaN
8576,0,ZTLVC029,1.016944,1.016944,1772829612.701468,ZTLVC029.edi,0.00001,0.063291,True,False,NaN
8577,0,ZTLVC030,1.016944,1.016944,1772829612.714315,ZTLVC030.edi,0.00001,0.063291,True,False,NaN


In [22]:
df.note.unique()

array([nan], dtype=object)

In [23]:
adf = df.loc[df.note.notnull(), ["filename", "note"]]

for index, row in adf.iterrows():
    print(f"{row.filename}: {row.note}")


In [24]:
import geopandas as gpd

In [25]:
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326")

In [26]:
gdf.to_file(edi_path.joinpath("edi_file_summary_2026_03_09.shp"), driver="ESRI Shapefile")
gdf.to_file(edi_path.joinpath("edi_file_summary_2026_03_09.geojson"), driver="GeoJSON")